## Store Sales - Time Series Forecasting


This project forecasts daily unit sales for thousands of product families at Favorita grocery stores in Ecuador. It is a time series forecasting problem with many series (54 stores x 33 product families), where the model must predict the 16 days that follow the end of the training data.

## Approach
1. Load the data and explore the date range, stores, and product families
2. Create calendar features (day of week, day of month, month, year, payday effects)
3. Validate by time: train on older dates and validate on the most recent 16 days (never with a random split)
4. Train a baseline gradient boosting model on log-transformed sales
5. Generate predictions and create the submission file
6. Submit to Kaggle and record the score

In [ ]:
import os
from getpass import getpass

os.environ["KAGGLE_API_TOKEN"] = getpass("Paste your Kaggle API token and press Enter: ")

!pip install -q -U kaggle
!kaggle competitions download -c store-sales-time-series-forecasting
!unzip -oq store-sales-time-series-forecasting.zip

In [2]:
import pandas as pd
import numpy as np

train = pd.read_csv("train.csv", parse_dates=["date"])
test = pd.read_csv("test.csv", parse_dates=["date"])

In [3]:
train.shape, test.shape

((3000888, 6), (28512, 5))

In [4]:
train["date"].min(), train["date"].max()

(Timestamp('2013-01-01 00:00:00'), Timestamp('2017-08-15 00:00:00'))

In [5]:
test["date"].min(), test["date"].max()

(Timestamp('2017-08-16 00:00:00'), Timestamp('2017-08-31 00:00:00'))

In [6]:
train.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3000888 entries, 0 to 3000887
Data columns (total 6 columns):
 #   Column       Dtype         
---  ------       -----         
 0   id           int64         
 1   date         datetime64[ns]
 2   store_nbr    int64         
 3   family       object        
 4   sales        float64       
 5   onpromotion  int64         
dtypes: datetime64[ns](1), float64(1), int64(3), object(1)
memory usage: 137.4+ MB


In [7]:
test.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 28512 entries, 0 to 28511
Data columns (total 5 columns):
 #   Column       Non-Null Count  Dtype         
---  ------       --------------  -----         
 0   id           28512 non-null  int64         
 1   date         28512 non-null  datetime64[ns]
 2   store_nbr    28512 non-null  int64         
 3   family       28512 non-null  object        
 4   onpromotion  28512 non-null  int64         
dtypes: datetime64[ns](1), int64(3), object(1)
memory usage: 1.1+ MB


In [8]:
train.sample(1)

,id,date,store_nbr,family,sales,onpromotion
1154834,1154834,2014-10-12,11,SEAFOOD,14.0,0


In [9]:
test.sample(1)

,id,date,store_nbr,family,onpromotion
7242,3008130,2017-08-20,12,HOME AND KITCHEN I,0


In [10]:
train.head(3)

,id,date,store_nbr,family,sales,onpromotion
0,0,2013-01-01,1,AUTOMOTIVE,0.0,0
1,1,2013-01-01,1,BABY CARE,0.0,0
2,2,2013-01-01,1,BEAUTY,0.0,0


In [11]:
train["family"].nunique()

33

In [12]:
train["store_nbr"].nunique()

54

In [13]:
test_ids = test["id"]
train = train[train["date"] >= "2016-01-01"]

In [14]:
train["dow"], train["day"], train["month"], train["year"] = train["date"].dt.dayofweek, train["date"].dt.day, train["date"].dt.month, train["date"].dt.year

In [15]:
test["dow"], test["day"], test["month"], test["year"] = test["date"].dt.dayofweek, test["date"].dt.day, test["date"].dt.month, test["date"].dt.year

In [16]:
train = pd.get_dummies(train, columns=["family"])
test = pd.get_dummies(test, columns=["family"])
train, test = train.align(test, join="left", axis=1, fill_value=0)

In [17]:
train.shape, test.shape

((1054944, 42), (28512, 42))

In [19]:
cutoff = train["date"].max() - pd.Timedelta(days=16)
tr = train[train["date"] <= cutoff]
val = train[train["date"] > cutoff]

In [20]:
x_train = tr.drop(columns=["id", "date", "sales"])
y_train = np.log1p(tr["sales"])
x_val = val.drop(columns=["id", "date", "sales"])
y_val = np.log1p(val["sales"])

In [21]:
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.metrics import mean_squared_error

model = HistGradientBoostingRegressor(max_iter=300, learning_rate=0.1, random_state=42)
model.fit(x_train, y_train)

pred = np.clip(model.predict(x_val), 0, None)
rmsle = mean_squared_error(y_val, pred) ** 0.5
print("Validation RMSLE:", round(rmsle, 4))

Validation RMSLE: 0.5245


In [22]:
x = train.drop(columns=["id", "date", "sales"])
y = np.log1p(train["sales"])

model.fit(x, y)

x_test = test.drop(columns=["id", "date", "sales"])
pred_test = np.expm1(np.clip(model.predict(x_test), 0, None))

pd.DataFrame({"id": test_ids, "sales": pred_test}).to_csv("submission.csv", index=False)

In [23]:
!kaggle competitions submit -c store-sales-time-series-forecasting -f submission.csv -m "HistGradientBoosting calendar baseline"

100% 727k/727k [00:00<00:00, 2.55MB/s]
4 submissions remaining today.
Successfully submitted to Store Sales - Time Series Forecasting

In [24]:
import pickle, os, sklearn
print("sklearn:", sklearn.__version__)
pickle.dump(model, open("sales_model.pkl", "wb"))
print("size MB:", round(os.path.getsize("sales_model.pkl") / 1e6, 2))

sklearn: 1.6.1
size MB: 1.08
